In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(42)

In [3]:
# num_experts
# top_k
# d_model
# dropout

In [2]:
class Expert(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.ffwd = nn.Sequential(
        nn.Linear(config.d_model, 4*config.d_model),
        nn.ReLU(),
        nn.Linear(4*config.d_model, config.d_model),
        nn.Dropout(config.dropout)
    )

  def forward(self, x):
    return self.ffwd(x)

In [5]:
class TopKRouter(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.top_k = config.top_k
    self.linear = nn.Linear(config.d_model, config.num_experts) # routing matrix
    self.noise_linear = nn.Linear(config.d_model, config.num_experts)

  def forward(self, x):
    # x is the output tensor from the mha block (x: (B, T, C))
    logits = self.linear(x) # expert selecter matric
    noise_logits = self.noise_linear(x)

    # softplus is a smooth version of ReLU and is always positive
    noise = torch.randn_like(logits) * F.softplus(noise_logits) # (B, T, num_experts)
    noisy_logits = logits + noise

    top_k_logits, indices = logits.top_k(self.top_k, dim=-1)
    zeros_matrix = torch.full_like(top_k_logits, float("-inf"))
    sparse_logits = zeros_matrix.scatter(dim=-1, index=indices, src=top_k_logits)
    router_output = F.softmax(sparse_logits, dim=-1)
    return router_output, indices